In [ ]:
# Post-hoc distribution check for Step6 cross-scene probing.
#
# Uses the Step5 pair-level Qwen2.5-3B/Qwen2.5-0.5B output and applies the same
# filtering logic as step6_cross_scs:
#   - pair samples
#   - explicit_direct evidence
#   - relation labels
#   - non-none directional relations
#
# Train scenes are kitchen, living_room, and bedroom; bathroom is held out
# as the test scene type. The goal here is to inspect probe-pair counts,
# directed object-type pair coverage, and label/evidence distributions,
# not to train probes or analyze hidden-state vectors.

import os
import re
import gc
import json
import math
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Set, Optional

import numpy as np
import pandas as pd
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab. Drive mount skipped.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B"
MODEL_TAG = "qwen2_5_0_3b"
SAMPLE_FAMILY = "pair"

STEP5_PT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_full/"
    "data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b"
)

# Only read pair-level probe samples, not composition samples.
PT_FILE_PATTERN = "*_step4_probe_samples_diverse_step5_hs_*.pt"

ANALYSIS_OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_full/"
    "data_prepare_step1_3/analysis/"
    "ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs"
)

os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)


USE_LOCAL_PT_CACHE = True

LOCAL_PT_CACHE_DIR = os.path.join(
    "/content",
    f"step5_pt_cache_{MODEL_TAG}_probe_pair_dist",
)

DELETE_LOCAL_PT_AFTER_LOAD = True

os.makedirs(LOCAL_PT_CACHE_DIR, exist_ok=True)


EXPLICIT_DIRECT = "explicit_direct"

LABEL_FIELD = "relation"

# Match step6_cross_scs:
# current experiment keeps only explicit_direct pair-level samples.
KEEP_EVIDENCE_TYPES = {
    EXPLICIT_DIRECT,
}

DROP_NONE_LABEL = True
DROP_EMPTY_LABEL = True
NONE_RELATION_LABEL = "none"

# Match STEP6_LABEL_SUBSET_MODE = "all_directional".
# This excludes "near" and keeps only non-none directional relations.
DIRECTIONAL_RELATIONS = {
    "in",
    "contains",
    "on",
    "supports",
    "left_of",
    "right_of",
    "above",
    "below",
}

ALLOWED_LABELS = set(DIRECTIONAL_RELATIONS)

ACTIVE_TRAIN_SCENES = (
    [f"FloorPlan{i}" for i in range(1, 31)]
    + [f"FloorPlan{i}" for i in range(201, 231)]
    + [f"FloorPlan{i}" for i in range(301, 331)]
)

ACTIVE_TEST_SCENES = [
    f"FloorPlan{i}" for i in range(401, 431)
]

TARGET_SCENE_TYPE = "bathroom"

# Make sure the filtered data matches the setup expected by step6_cross_scs.
if SAMPLE_FAMILY != "pair":
    raise ValueError(f"step6_cross_scs expects SAMPLE_FAMILY='pair', got {SAMPLE_FAMILY!r}")

expected_evidence = {EXPLICIT_DIRECT}
if KEEP_EVIDENCE_TYPES != expected_evidence:
    raise ValueError(
        f"step6_cross_scs expects KEEP_EVIDENCE_TYPES={expected_evidence}, "
        f"got {KEEP_EVIDENCE_TYPES}"
    )

if LABEL_FIELD != "relation":
    raise ValueError(f"step6_cross_scs expects LABEL_FIELD='relation', got {LABEL_FIELD!r}")

if not DROP_NONE_LABEL:
    raise ValueError("step6_cross_scs expects DROP_NONE_LABEL=True.")

if not DROP_EMPTY_LABEL:
    raise ValueError("step6_cross_scs expects DROP_EMPTY_LABEL=True.")

if set(ACTIVE_TRAIN_SCENES) & set(ACTIVE_TEST_SCENES):
    raise ValueError("ACTIVE_TRAIN_SCENES and ACTIVE_TEST_SCENES must not overlap.")

if len(ACTIVE_TRAIN_SCENES) != 90:
    raise ValueError(f"Expected 90 train scenes, got {len(ACTIVE_TRAIN_SCENES)}")

if len(ACTIVE_TEST_SCENES) != 30:
    raise ValueError(f"Expected 30 test scenes, got {len(ACTIVE_TEST_SCENES)}")


print("=" * 100)
print("SINGLE-MODEL STEP6-CROSS-SCS ANALYSIS CONFIGURATION")
print("=" * 100)

print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("SAMPLE_FAMILY:", SAMPLE_FAMILY)
print("STEP5_PT_DIR:", STEP5_PT_DIR)
print("PT_FILE_PATTERN:", PT_FILE_PATTERN)
print("ANALYSIS_OUTPUT_DIR:", ANALYSIS_OUTPUT_DIR)

print("\nLocal cache:")
print("USE_LOCAL_PT_CACHE:", USE_LOCAL_PT_CACHE)
print("LOCAL_PT_CACHE_DIR:", LOCAL_PT_CACHE_DIR)
print("DELETE_LOCAL_PT_AFTER_LOAD:", DELETE_LOCAL_PT_AFTER_LOAD)

print("\nFiltering:")
print("LABEL_FIELD:", LABEL_FIELD)
print("KEEP_EVIDENCE_TYPES:", sorted(KEEP_EVIDENCE_TYPES))
print("DROP_NONE_LABEL:", DROP_NONE_LABEL)
print("DROP_EMPTY_LABEL:", DROP_EMPTY_LABEL)
print("NONE_RELATION_LABEL:", NONE_RELATION_LABEL)
print("ALLOWED_LABELS:", sorted(ALLOWED_LABELS))

print("\nScene split:")
print("Number of train scenes:", len(ACTIVE_TRAIN_SCENES))
print("First train scenes:", ACTIVE_TRAIN_SCENES[:5])
print("Last train scenes:", ACTIVE_TRAIN_SCENES[-5:])
print("Number of test scenes:", len(ACTIVE_TEST_SCENES))
print("First test scenes:", ACTIVE_TEST_SCENES[:5])
print("Last test scenes:", ACTIVE_TEST_SCENES[-5:])
print("TARGET_SCENE_TYPE:", TARGET_SCENE_TYPE)

SINGLE-MODEL STEP6-CROSS-SCS ANALYSIS CONFIGURATION
MODEL_NAME: Qwen/Qwen2.5-3B
MODEL_TAG: qwen2_5_0_3b
SAMPLE_FAMILY: pair
STEP5_PT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b
PT_FILE_PATTERN: *_step4_probe_samples_diverse_step5_hs_*.pt
ANALYSIS_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/analysis/ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs

Local cache:
USE_LOCAL_PT_CACHE: True
LOCAL_PT_CACHE_DIR: /content/step5_pt_cache_qwen2_5_0_3b_probe_pair_dist
DELETE_LOCAL_PT_AFTER_LOAD: True

Filtering:
LABEL_FIELD: relation
KEEP_EVIDENCE_TYPES: ['explicit_direct']
DROP_NONE_LABEL: True
DROP_EMPTY_LABEL: True
NONE_RELATION_LABEL: none
ALLOWED_LABELS: ['above', 'below', 'contains', 'in', 'left_of', 'on', 'right_of', 'supports']

Scene split:
Number of train scenes: 90
First train scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5']
Last train scenes

In [ ]:
# Helper functions

import shutil
import time


def make_json_safe(obj):
    """
    Convert numpy / torch / set objects into JSON-serializable structures.
    """
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, set):
        return sorted(make_json_safe(v) for v in obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, torch.Tensor):
        return {
            "__tensor_shape__": list(obj.shape),
            "__tensor_dtype__": str(obj.dtype),
        }

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    return obj


def natural_sort_key(text: str):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(text))
    ]


def infer_scene_type_from_floorplan(scene: Any) -> str:
    """
    Infer iTHOR scene type from FloorPlan id.

    iTHOR convention:
        FloorPlan1-30      kitchen
        FloorPlan201-230   living_room
        FloorPlan301-330   bedroom
        FloorPlan401-430   bathroom
    """
    if not isinstance(scene, str):
        return "unknown"

    m = re.search(r"FloorPlan(\d+)", scene)

    if not m:
        return "unknown"

    n = int(m.group(1))

    if 1 <= n <= 30:
        return "kitchen"

    if 201 <= n <= 230:
        return "living_room"

    if 301 <= n <= 330:
        return "bedroom"

    if 401 <= n <= 430:
        return "bathroom"

    return "unknown"


def normalize_missing(value: Any) -> str:
    """
    Convert missing / empty metadata values to stable strings.
    """
    if value is None:
        return "__MISSING__"

    if isinstance(value, float) and math.isnan(value):
        return "__MISSING__"

    if isinstance(value, str) and value.strip() == "":
        return "__EMPTY__"

    return str(value)


def safe_ratio(numerator: float, denominator: float) -> float:
    if denominator == 0:
        return 0.0
    return float(numerator) / float(denominator)


def keep_record_for_step6(
    rec: Dict[str, Any],
    label_field: str,
    keep_evidence_types: Set[str],
    drop_none_label: bool,
    drop_empty_label: bool,
    none_label: str,
    allowed_labels: Optional[Set[str]] = None,
) -> bool:
    """
    Reproduce the filtering logic used by step6_cross_scs.

    Order:
      1. keep only allowed evidence types;
      2. drop empty labels if requested;
      3. cast label to str;
      4. drop none label if requested;
      5. keep only allowed labels if allowed_labels is not None.
    """
    evidence_type = rec.get("evidence_type")

    if keep_evidence_types and evidence_type not in keep_evidence_types:
        return False

    label = rec.get(label_field)

    if drop_empty_label and (label is None or str(label).strip() == ""):
        return False

    label = str(label)

    if drop_none_label and label == none_label:
        return False

    if allowed_labels is not None and label not in allowed_labels:
        return False

    return True


def is_drive_transport_error(exc: BaseException) -> bool:
    """
    Detect common Google Drive FUSE transport errors in Colab.
    """
    msg = str(exc)

    return (
        "Transport endpoint is not connected" in msg
        or "Software caused connection abort" in msg
        or "Input/output error" in msg
        or "Connection aborted" in msg
        or "ConnectionAbortedError" in msg
    )


def copy_pt_to_local_cache(
    src_path: str,
    local_cache_dir: str,
    max_retries: int = 3,
    sleep_seconds: int = 5,
) -> str:
    """
    Copy a Step5 .pt file from Google Drive to local /content cache.

    This avoids torch.load() directly streaming from /content/drive,
    which can trigger Google Drive FUSE transport errors.
    """
    os.makedirs(local_cache_dir, exist_ok=True)

    local_path = os.path.join(
        local_cache_dir,
        os.path.basename(src_path),
    )

    if os.path.exists(local_path):
        try:
            src_size = os.path.getsize(src_path)
            local_size = os.path.getsize(local_path)

            if src_size == local_size and local_size > 0:
                return local_path

            os.remove(local_path)

        except Exception:
            try:
                os.remove(local_path)
            except Exception:
                pass

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(
                f"    Copying to local cache "
                f"(attempt {attempt}/{max_retries}): {os.path.basename(src_path)}",
                flush=True,
            )

            shutil.copy2(src_path, local_path)

            src_size = os.path.getsize(src_path)
            local_size = os.path.getsize(local_path)

            if src_size != local_size:
                raise IOError(
                    f"Incomplete copy: src_size={src_size}, local_size={local_size}"
                )

            return local_path

        except Exception as exc:
            last_error = exc

            if os.path.exists(local_path):
                try:
                    os.remove(local_path)
                except Exception:
                    pass

            print(
                f"    Copy failed attempt {attempt}/{max_retries}\n"
                f"      src: {src_path}\n"
                f"      error: {repr(exc)}",
                flush=True,
            )

            if attempt < max_retries:
                time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Failed to copy file to local cache after {max_retries} attempts:\n"
        f"{src_path}\nLast error: {repr(last_error)}"
    )


def safe_torch_load_from_local_cache(
    drive_pt_path: str,
    local_cache_dir: str,
    map_location: str = "cpu",
    delete_after_load: bool = True,
):
    """
    Load a .pt file robustly by first copying it from Google Drive
    to /content local cache, then calling torch.load() on the local file.
    """
    local_path = copy_pt_to_local_cache(
        src_path=drive_pt_path,
        local_cache_dir=local_cache_dir,
    )

    try:
        payload = torch.load(local_path, map_location=map_location)
        return payload

    finally:
        if delete_after_load and os.path.exists(local_path):
            try:
                os.remove(local_path)
            except Exception as exc:
                print(
                    f"Warning: failed to delete local cache file {local_path}: {exc}",
                    flush=True,
                )


# Only keep lightweight metadata fields.
# Not needed: Hidden-state features, token ids, decoded tokens, offset mappings, and masks
METADATA_KEYS_TO_KEEP = {
    "sample_id",
    "sample_type",
    "pair_group_id",

    "scene",
    "scene_type",
    "paragraph_id",
    "chunk_id",
    "cluster_id",
    "generation_mode",

    "subject_alias",
    "object_alias",
    "subject_uid",
    "object_uid",
    "subject_id",
    "object_id",
    "subject_type",
    "object_type",

    "relation",
    "relation_label",
    "has_relation_label",
    "label_is_none",

    "evidence_type",
    "pair_evidence_type",
    "target_pair_evidence_type",
    "probe_direction_relative_to_text",
    "is_inverse_of_text_relation",
    "relation_source",
}

In [ ]:
# Path sanity check for single-model Step5 input

if not KEEP_EVIDENCE_TYPES:
    print(
        "Warning: KEEP_EVIDENCE_TYPES is empty. "
        "No evidence-type filtering will be applied."
    )

if KEEP_EVIDENCE_TYPES != {EXPLICIT_DIRECT}:
    raise ValueError(
        "This analysis is intended to match step6_cross_scs, "
        f"which requires explicit_direct only. Got: {sorted(KEEP_EVIDENCE_TYPES)}"
    )

if ALLOWED_LABELS is None:
    raise ValueError(
        "This analysis is intended to match step6_cross_scs all_directional mode; "
        "ALLOWED_LABELS must not be None."
    )

if not os.path.exists(STEP5_PT_DIR):
    raise FileNotFoundError(f"STEP5_PT_DIR does not exist: {STEP5_PT_DIR}")

pt_files = sorted(
    Path(STEP5_PT_DIR).glob(PT_FILE_PATTERN),
    key=lambda p: natural_sort_key(str(p)),
)

print("=" * 100)
print("Discovered Step5 .pt files")
print("=" * 100)

print("MODEL_TAG:", MODEL_TAG)
print("STEP5_PT_DIR:", STEP5_PT_DIR)
print("PT_FILE_PATTERN:", PT_FILE_PATTERN)
print("Number of .pt files:", len(pt_files))

print("\nFirst files:")
for p in pt_files[:5]:
    print("-", p)

if not pt_files:
    raise FileNotFoundError(
        f"No .pt files found in:\n{STEP5_PT_DIR}\nPattern: {PT_FILE_PATTERN}"
    )

if not ACTIVE_TRAIN_SCENES:
    print("Warning: ACTIVE_TRAIN_SCENES is empty. Train-coverage analysis will be skipped.")

if not ACTIVE_TEST_SCENES:
    print("Warning: ACTIVE_TEST_SCENES is empty. Test-scene analysis will still use scene_type.")

Discovered Step5 .pt files
MODEL_TAG: qwen2_5_0_3b
STEP5_PT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b
PT_FILE_PATTERN: *_step4_probe_samples_diverse_step5_hs_*.pt
Number of .pt files: 118

First files:
- /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b/FloorPlan1_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b/FloorPlan2_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b/FloorPlan3_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b/FloorPlan4_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- /content/drive/MyDrive/Colab Notebook

In [ ]:
pt_paths = sorted(
    Path(STEP5_PT_DIR).glob(PT_FILE_PATTERN),
    key=lambda p: natural_sort_key(str(p)),
)

if not pt_paths:
    raise FileNotFoundError(
        f"No Step5 .pt files found.\n"
        f"Input dir: {STEP5_PT_DIR}\n"
        f"Pattern: {PT_FILE_PATTERN}"
    )

pt_file_records = []

for pt_path in pt_paths:
    pt_file_records.append({
        "model_tag": MODEL_TAG,
        "model_name": MODEL_NAME,
        "sample_family": SAMPLE_FAMILY,
        "pt_dir": STEP5_PT_DIR,
        "path": str(pt_path),
        "relative_path": os.path.relpath(pt_path, STEP5_PT_DIR),
        "file_size_mb": os.path.getsize(pt_path) / 1024**2,
    })

pt_file_df = pd.DataFrame(pt_file_records)

print("=" * 100)
print("Step5 .pt files selected for analysis")
print("=" * 100)

print("Total .pt files:", len(pt_file_df))
print("Total size MB:", round(pt_file_df["file_size_mb"].sum(), 2))

print("\nPreview:")
display(pt_file_df.head(10))

Step5 .pt files selected for analysis
Total .pt files: 118
Total size MB: 111180.2

Preview:


,model_tag,model_name,sample_family,pt_dir,path,relative_path,file_size_mb
0,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan1_step4_probe_samples_diverse_step5_h...,1935.310853
1,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan2_step4_probe_samples_diverse_step5_h...,1347.067529
2,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan3_step4_probe_samples_diverse_step5_h...,1039.303811
3,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan4_step4_probe_samples_diverse_step5_h...,1672.736596
4,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan5_step4_probe_samples_diverse_step5_h...,1829.538796
5,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan6_step4_probe_samples_diverse_step5_h...,1604.126885
6,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan7_step4_probe_samples_diverse_step5_h...,1601.228677
7,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan8_step4_probe_samples_diverse_step5_h...,1511.286599
8,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan9_step4_probe_samples_diverse_step5_h...,1572.561654
9,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,/content/drive/MyDrive/Colab Notebooks/linear_...,/content/drive/MyDrive/Colab Notebooks/linear_...,FloorPlan10_step4_probe_samples_diverse_step5_...,1742.124701


In [ ]:
# Build a metadata table from the Step5 .pt files using the Step6 filters.
#
# To avoid keeping hidden-state arrays in memory, load one file at a time
# and keep only the fields needed for the distribution checks.
#
# If USE_LOCAL_PT_CACHE is enabled, copy each file to /content first and
# load it from there. This is more stable than calling torch.load() on the
# Google Drive path directly.

all_rows = []
source_file_summaries = []

raw_record_count_total = 0
kept_record_count_total = 0
dropped_record_count_total = 0

if USE_LOCAL_PT_CACHE:
    os.makedirs(LOCAL_PT_CACHE_DIR, exist_ok=True)

print("=" * 100)
print("Scanning Step5 payloads")
print("=" * 100)

for file_i, item in enumerate(pt_file_records, start=1):
    pt_path = item["path"]
    model_tag = item["model_tag"]
    model_name = item["model_name"]
    sample_family = item["sample_family"]
    relative_path = item["relative_path"]

    print(
        f"\n[{file_i}/{len(pt_file_records)}] "
        f"file={relative_path}",
        flush=True,
    )

    try:
        if USE_LOCAL_PT_CACHE:
            payload = safe_torch_load_from_local_cache(
                drive_pt_path=pt_path,
                local_cache_dir=LOCAL_PT_CACHE_DIR,
                map_location="cpu",
                delete_after_load=DELETE_LOCAL_PT_AFTER_LOAD,
            )
        else:
            payload = torch.load(pt_path, map_location="cpu")

    except Exception as exc:
        print(
            f"\nFailed to load Step5 payload:\n"
            f"  file: {pt_path}\n"
            f"  error: {repr(exc)}",
            flush=True,
        )

        if is_drive_transport_error(exc):
            print(
                "\nThis looks like a Google Drive transport error. "
                "The local-cache path is already enabled if USE_LOCAL_PT_CACHE=True. "
                "If the copy itself fails, remount Drive or restart the Colab runtime.",
                flush=True,
            )

        raise

    if "records" not in payload:
        raise KeyError(
            f"Expected key 'records' in Step5 payload.\n"
            f"path={pt_path}\n"
            f"payload_keys={sorted(list(payload.keys()))}"
        )

    records = payload["records"]
    raw_record_count_total += len(records)

    kept_in_file = 0
    dropped_in_file = 0
    raw_evidence_counter = Counter()
    kept_evidence_counter = Counter()
    raw_label_counter = Counter()
    kept_label_counter = Counter()

    for local_record_idx, rec in enumerate(records):
        raw_evidence_counter[rec.get("evidence_type")] += 1
        raw_label_counter[rec.get(LABEL_FIELD)] += 1

        keep = keep_record_for_step6(
            rec=rec,
            label_field=LABEL_FIELD,
            keep_evidence_types=KEEP_EVIDENCE_TYPES,
            drop_none_label=DROP_NONE_LABEL,
            drop_empty_label=DROP_EMPTY_LABEL,
            none_label=NONE_RELATION_LABEL,
            allowed_labels=ALLOWED_LABELS,
        )

        if not keep:
            dropped_in_file += 1
            continue

        row = {k: rec.get(k) for k in METADATA_KEYS_TO_KEEP}

        # Provenance fields
        row["_analysis_model_tag"] = model_tag
        row["_analysis_model_name"] = model_name
        row["_analysis_sample_family"] = sample_family
        row["_source_step5_pt_file"] = relative_path
        row["_source_step5_absolute_pt_file"] = pt_path
        row["_source_local_record_idx"] = int(local_record_idx)

        all_rows.append(row)

        kept_in_file += 1
        kept_evidence_counter[rec.get("evidence_type")] += 1
        kept_label_counter[rec.get(LABEL_FIELD)] += 1

    dropped_record_count_total += dropped_in_file
    kept_record_count_total += kept_in_file

    source_file_summaries.append({
        "model_tag": model_tag,
        "model_name": model_name,
        "sample_family": sample_family,
        "pt_file": relative_path,
        "absolute_pt_file": pt_path,
        "file_size_mb": item["file_size_mb"],

        "payload_source_file": payload.get("source_file"),
        "payload_scene": payload.get("scene"),
        "payload_source_type": payload.get("source_type"),
        "payload_model_name": payload.get("model_name"),
        "payload_model_tag": payload.get("model_tag"),
        "payload_sample_family": payload.get("sample_family"),

        "num_records_raw": len(records),
        "num_records_kept_after_step6_filter": kept_in_file,
        "num_records_dropped_after_step6_filter": dropped_in_file,

        "raw_evidence_counts": dict(raw_evidence_counter),
        "kept_evidence_counts": dict(kept_evidence_counter),
        "raw_label_counts": dict(raw_label_counter),
        "kept_label_counts": dict(kept_label_counter),

        "payload_keys": sorted(list(payload.keys())),
    })

    print(
        f"    raw={len(records)} | "
        f"kept={kept_in_file} | "
        f"dropped={dropped_in_file} | "
        f"total_kept={kept_record_count_total}",
        flush=True,
    )

    del payload
    del records
    gc.collect()

df = pd.DataFrame(all_rows)

print("\n" + "=" * 100)
print("Finished metadata scan")
print("=" * 100)
print("Raw Step5 records:", raw_record_count_total)
print("Kept after Step6-compatible filter:", kept_record_count_total)
print("Dropped by Step6-compatible filter:", dropped_record_count_total)
print("DataFrame shape:", df.shape)

if df.empty:
    raise ValueError(
        "No records remain after Step6-compatible filtering. "
        "Check KEEP_EVIDENCE_TYPES, LABEL_FIELD, DROP_NONE_LABEL, ALLOWED_LABELS, and input dir."
    )

display(df.head())

Scanning Step5 payloads

[1/118] file=FloorPlan1_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
    Copying to local cache (attempt 1/3): FloorPlan1_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
    raw=2666 | kept=337 | dropped=2329 | total_kept=337

[2/118] file=FloorPlan2_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
    Copying to local cache (attempt 1/3): FloorPlan2_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
    Copy failed attempt 1/3
      src: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b/FloorPlan2_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
      error: OSError(107, 'Transport endpoint is not connected')
    Copying to local cache (attempt 2/3): FloorPlan2_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
    raw=1856 | kept=243 | dropped=1613 | total_kept=580

[3/118] file=FloorPlan3_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
    Copying to local cache (attempt 1/3): FloorPla

,object_id,generation_mode,object_alias,relation_source,sample_type,target_pair_evidence_type,sample_id,pair_group_id,evidence_type,label_is_none,...,object_type,subject_type,subject_alias,object_uid,_analysis_model_tag,_analysis_model_name,_analysis_sample_family,_source_step5_pt_file,_source_step5_absolute_pt_file,_source_local_record_idx
0,Pot|-01.22|+00.90|-02.36,diverse,pot_1,direct,pair,None,s4_e313e62718eaedc3182f6f70,pg_e41d676be99a9d275cdf,explicit_direct,False,...,Pot,Cabinet,cabinet_1,pot_1,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,FloorPlan1_step4_probe_samples_diverse_step5_h...,/content/drive/MyDrive/Colab Notebooks/linear_...,9
1,WineBottle|-01.00|+01.65|-02.58,diverse,wine_bottle_1,direct,pair,None,s4_da271e7106d2190831112344,pg_65b04d926452d11e3e80,explicit_direct,False,...,WineBottle,Cabinet,cabinet_1,wine_bottle_1,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,FloorPlan1_step4_probe_samples_diverse_step5_h...,/content/drive/MyDrive/Colab Notebooks/linear_...,10
2,DishSponge|-01.94|+00.75|-01.71,diverse,dish_sponge_1,direct,pair,None,s4_024f4dea8a98966d4b36bbc7,pg_c5f2d700f4ca855a05e5,explicit_direct,False,...,DishSponge,Cabinet,cabinet_2,dish_sponge_1,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,FloorPlan1_step4_probe_samples_diverse_step5_h...,/content/drive/MyDrive/Colab Notebooks/linear_...,14
3,CounterTop|-01.87|+00.95|-01.21,diverse,counter_top_1,direct,pair,None,s4_945f4318fbe7030012cefde4,pg_43869b3d20c2e0f2593d,explicit_direct,False,...,CounterTop,CoffeeMachine,coffee_machine_1,counter_top_1,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,FloorPlan1_step4_probe_samples_diverse_step5_h...,/content/drive/MyDrive/Colab Notebooks/linear_...,24
4,DishSponge|-01.94|+00.75|-01.71,diverse,dish_sponge_1,direct,pair,None,s4_0c64bf21282fa5d69a0d44f2,pg_028c710072213e489e86,explicit_direct,False,...,DishSponge,CounterTop,counter_top_1,dish_sponge_1,qwen2_5_0_3b,Qwen/Qwen2.5-3B,pair,FloorPlan1_step4_probe_samples_diverse_step5_h...,/content/drive/MyDrive/Colab Notebooks/linear_...,36


In [ ]:
required_columns = [
    "sample_id",
    "scene",
    LABEL_FIELD,
    "evidence_type",
    "subject_type",
    "object_type",
]

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise KeyError(
        "Missing required metadata columns:\n"
        + json.dumps(missing_required_columns, indent=2, ensure_ascii=False)
        + "\nAvailable columns:\n"
        + json.dumps(sorted(df.columns.tolist()), indent=2, ensure_ascii=False)
    )

if "scene_type" not in df.columns:
    df["scene_type"] = None

df["scene_type_inferred"] = df["scene"].apply(infer_scene_type_from_floorplan)

df["scene_type"] = df["scene_type"].where(
    df["scene_type"].notna() & (df["scene_type"].astype(str).str.strip() != ""),
    df["scene_type_inferred"],
)

optional_columns = [
    "paragraph_id",
    "chunk_id",
    "cluster_id",
    "subject_uid",
    "object_uid",
    "subject_id",
    "object_id",
    "relation",
    "relation_label",
    "pair_evidence_type",
]

for col in optional_columns:
    if col not in df.columns:
        df[col] = None

for col in [
    "scene",
    "scene_type",
    "paragraph_id",
    "subject_uid",
    "object_uid",
    "subject_id",
    "object_id",
    "subject_type",
    "object_type",
    "relation",
    "relation_label",
    "evidence_type",
    "pair_evidence_type",
    LABEL_FIELD,
]:
    if col not in df.columns:
        df[col] = None

    df[col] = df[col].apply(normalize_missing)

# Probe label

df["probe_label"] = df[LABEL_FIELD].apply(normalize_missing)

observed_evidence = set(df["evidence_type"].astype(str).unique())
expected_evidence = {EXPLICIT_DIRECT}

if observed_evidence != expected_evidence:
    raise ValueError(
        "Filtered records are not aligned with step6_cross_scs. "
        f"Expected evidence types {sorted(expected_evidence)}, "
        f"observed {sorted(observed_evidence)}."
    )

if DROP_NONE_LABEL and NONE_RELATION_LABEL in set(df["probe_label"].astype(str).unique()):
    raise ValueError(
        f"Filtered records still contain NONE_RELATION_LABEL={NONE_RELATION_LABEL!r}."
    )

if ALLOWED_LABELS is not None:
    observed_labels = set(df["probe_label"].astype(str).unique())
    disallowed_labels = sorted(observed_labels - ALLOWED_LABELS)

    if disallowed_labels:
        raise ValueError(
            "Filtered records contain labels outside step6_cross_scs ALLOWED_LABELS: "
            f"{disallowed_labels}"
        )

df["probe_record_key"] = df["sample_id"].astype(str)

df["probe_record_key_strict"] = (
    df["scene"].astype(str)
    + "|paragraph=" + df["paragraph_id"].astype(str)
    + "|subj=" + df["subject_uid"].astype(str)
    + "|obj=" + df["object_uid"].astype(str)
    + "|label=" + df["probe_label"].astype(str)
    + "|evi=" + df["evidence_type"].astype(str)
)


df["directed_uid_pair"] = (
    df["scene"].astype(str)
    + "|"
    + df["subject_uid"].astype(str)
    + "->"
    + df["object_uid"].astype(str)
)

df["directed_object_id_pair"] = (
    df["scene"].astype(str)
    + "|"
    + df["subject_id"].astype(str)
    + "->"
    + df["object_id"].astype(str)
)

df["directed_type_pair"] = (
    df["subject_type"].astype(str)
    + " -> "
    + df["object_type"].astype(str)
)

df["unordered_type_pair"] = df.apply(
    lambda r: " -- ".join(sorted([
        str(r["subject_type"]),
        str(r["object_type"]),
    ])),
    axis=1,
)


train_scene_set = set(ACTIVE_TRAIN_SCENES)
test_scene_set = set(ACTIVE_TEST_SCENES)

df["is_step6_train_scene"] = df["scene"].isin(train_scene_set)
df["is_step6_test_scene"] = df["scene"].isin(test_scene_set)

df["split"] = "outside_config_split"
df.loc[df["is_step6_train_scene"], "split"] = "train"
df.loc[df["is_step6_test_scene"], "split"] = "test"

overlap_rows = df[df["is_step6_train_scene"] & df["is_step6_test_scene"]]

if not overlap_rows.empty:
    raise ValueError(
        "Some records belong to both train and test scenes. "
        "Check ACTIVE_TRAIN_SCENES and ACTIVE_TEST_SCENES."
    )

if df["is_step6_train_scene"].sum() == 0:
    raise ValueError("No records assigned to train scenes.")

if df["is_step6_test_scene"].sum() == 0:
    raise ValueError("No records assigned to test scenes.")


duplicated_sample_ids = df["sample_id"].value_counts()
duplicated_sample_ids = duplicated_sample_ids[duplicated_sample_ids > 1]

print("=" * 100)
print("Metadata validation and canonical fields complete")
print("=" * 100)

print("MODEL_TAG:", MODEL_TAG)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Scene types:", sorted(df["scene_type"].unique().tolist()))
print("Splits:", df["split"].value_counts().to_dict())
print("Evidence types:", sorted(df["evidence_type"].unique().tolist()))
print("Probe labels:", sorted(df["probe_label"].unique().tolist()))
print("Duplicated sample_ids:", len(duplicated_sample_ids))

display(df[[
    "sample_id",
    "scene",
    "scene_type",
    "split",
    "subject_type",
    "object_type",
    "directed_type_pair",
    "probe_label",
    "evidence_type",
]].head())

Metadata validation and canonical fields complete
MODEL_TAG: qwen2_5_0_3b
Rows: 21916
Columns: 44
Scene types: ['bathroom', 'bedroom', 'kitchen', 'living_room']
Splits: {'train': 17226, 'test': 4690}
Evidence types: ['explicit_direct']
Probe labels: ['above', 'below', 'contains', 'in', 'left_of', 'on', 'right_of', 'supports']
Duplicated sample_ids: 0


,sample_id,scene,scene_type,split,subject_type,object_type,directed_type_pair,probe_label,evidence_type
0,s4_e313e62718eaedc3182f6f70,FloorPlan1,kitchen,train,Cabinet,Pot,Cabinet -> Pot,above,explicit_direct
1,s4_da271e7106d2190831112344,FloorPlan1,kitchen,train,Cabinet,WineBottle,Cabinet -> WineBottle,contains,explicit_direct
2,s4_024f4dea8a98966d4b36bbc7,FloorPlan1,kitchen,train,Cabinet,DishSponge,Cabinet -> DishSponge,contains,explicit_direct
3,s4_945f4318fbe7030012cefde4,FloorPlan1,kitchen,train,CoffeeMachine,CounterTop,CoffeeMachine -> CounterTop,on,explicit_direct
4,s4_0c64bf21282fa5d69a0d44f2,FloorPlan1,kitchen,train,CounterTop,DishSponge,CounterTop -> DishSponge,above,explicit_direct


In [ ]:
scene_type_summary = (
    df.groupby("scene_type")
      .agg(
          num_step6_filtered_records=("sample_id", "count"),
          num_unique_sample_ids=("sample_id", "nunique"),
          num_scenes=("scene", "nunique"),
          num_directed_uid_pairs=("directed_uid_pair", "nunique"),
          num_directed_object_id_pairs=("directed_object_id_pair", "nunique"),
          num_directed_type_pairs=("directed_type_pair", "nunique"),
          num_unordered_type_pairs=("unordered_type_pair", "nunique"),
          num_subject_types=("subject_type", "nunique"),
          num_object_types=("object_type", "nunique"),
          num_probe_labels=("probe_label", "nunique"),
          num_evidence_types=("evidence_type", "nunique"),
      )
      .reset_index()
      .sort_values("scene_type")
)

total_records = scene_type_summary["num_step6_filtered_records"].sum()
total_sample_ids = scene_type_summary["num_unique_sample_ids"].sum()

scene_type_summary["record_ratio_global"] = (
    scene_type_summary["num_step6_filtered_records"] / total_records
)

scene_type_summary["sample_id_ratio_global"] = (
    scene_type_summary["num_unique_sample_ids"] / total_sample_ids
)

scene_type_summary["avg_records_per_scene"] = (
    scene_type_summary["num_step6_filtered_records"] /
    scene_type_summary["num_scenes"].replace(0, np.nan)
)

scene_type_summary["avg_directed_type_pairs_per_scene"] = (
    scene_type_summary["num_directed_type_pairs"] /
    scene_type_summary["num_scenes"].replace(0, np.nan)
)

print("=" * 100)
print("Scene-type summary")
print("Filtering: explicit_direct only + non-none directional labels")
print("=" * 100)

display(scene_type_summary)

Scene-type summary
Filtering: explicit_direct only + non-none directional labels


,scene_type,num_step6_filtered_records,num_unique_sample_ids,num_scenes,num_directed_uid_pairs,num_directed_object_id_pairs,num_directed_type_pairs,num_unordered_type_pairs,num_subject_types,num_object_types,num_probe_labels,num_evidence_types,record_ratio_global,sample_id_ratio_global,avg_records_per_scene,avg_directed_type_pairs_per_scene
0,bathroom,4690,4690,30,1674,1683,510,314,36,36,8,1,0.213999,0.213999,156.333333,17.000000
1,bedroom,4527,4527,28,1963,1967,582,375,48,49,8,1,0.206561,0.206561,161.678571,20.785714
2,kitchen,8656,8656,30,4439,4488,1333,818,60,60,8,1,0.394963,0.394963,288.533333,44.433333
3,living_room,4043,4043,30,1378,1394,473,312,41,41,8,1,0.184477,0.184477,134.766667,15.766667


In [ ]:
scene_summary = (
    df.groupby(["scene_type", "scene", "split"])
      .agg(
          num_step6_filtered_records=("sample_id", "count"),
          num_unique_sample_ids=("sample_id", "nunique"),
          num_directed_uid_pairs=("directed_uid_pair", "nunique"),
          num_directed_object_id_pairs=("directed_object_id_pair", "nunique"),
          num_directed_type_pairs=("directed_type_pair", "nunique"),
          num_unordered_type_pairs=("unordered_type_pair", "nunique"),
          num_subject_types=("subject_type", "nunique"),
          num_object_types=("object_type", "nunique"),
          num_probe_labels=("probe_label", "nunique"),
      )
      .reset_index()
)


def scene_number_for_sort(scene):
    """
    Extract numeric FloorPlan id for stable sorting.
    Unknown / malformed scenes are placed at the end.
    """
    scene = str(scene)
    m = re.search(r"FloorPlan(\d+)", scene)
    if m:
        return int(m.group(1))
    return 10**9


scene_type_order = {
    "kitchen": 0,
    "living_room": 1,
    "bedroom": 2,
    "bathroom": 3,
    "unknown": 4,
}

scene_summary["_scene_type_sort"] = (
    scene_summary["scene_type"]
    .map(scene_type_order)
    .fillna(99)
    .astype(int)
)

scene_summary["_scene_number_sort"] = (
    scene_summary["scene"]
    .apply(scene_number_for_sort)
)

scene_summary = (
    scene_summary
    .sort_values(
        ["_scene_type_sort", "_scene_number_sort", "scene", "split"],
        ascending=[True, True, True, True],
    )
    .drop(columns=["_scene_type_sort", "_scene_number_sort"])
    .reset_index(drop=True)
)

print("=" * 100)
print("Scene-level summary")
print("=" * 100)

display(scene_summary)

Scene-level summary


,scene_type,scene,split,num_step6_filtered_records,num_unique_sample_ids,num_directed_uid_pairs,num_directed_object_id_pairs,num_directed_type_pairs,num_unordered_type_pairs,num_subject_types,num_object_types,num_probe_labels
0,kitchen,FloorPlan1,train,337,337,173,177,152,123,39,37,8
1,kitchen,FloorPlan2,train,243,243,110,112,99,87,27,27,8
2,kitchen,FloorPlan3,train,232,232,75,75,71,66,31,32,7
3,kitchen,FloorPlan4,train,287,287,173,175,162,135,34,34,8
4,kitchen,FloorPlan5,train,342,342,182,182,146,117,36,37,7
...,...,...,...,...,...,...,...,...,...,...,...,...
113,bathroom,FloorPlan426,test,135,135,42,42,37,29,16,16,6
114,bathroom,FloorPlan427,test,214,214,130,137,108,85,20,20,6
115,bathroom,FloorPlan428,test,136,136,47,48,45,39,17,15,8
116,bathroom,FloorPlan429,test,106,106,44,44,42,38,16,15,5


In [ ]:
df_target_all = df[df["scene_type"] == TARGET_SCENE_TYPE].copy()

# Prefer the Step6 test split if it is defined.
df_target_test = df[
    (df["scene_type"] == TARGET_SCENE_TYPE)
    & (df["is_step6_test_scene"])
].copy()

if not df_target_test.empty:
    df_target = df_target_test.copy()
    target_subset_mode = "target_scene_type_and_test_split"
else:
    df_target = df_target_all.copy()
    target_subset_mode = "target_scene_type_all_records"

print("=" * 100)
print(f"Target scene type: {TARGET_SCENE_TYPE}")
print("=" * 100)

print("Target subset mode:", target_subset_mode)
print("All target-scene records:", len(df_target_all))
print("Test target-scene records:", len(df_target_test))
print("Rows used for target analysis:", len(df_target))

if df_target.empty:
    raise ValueError(
        f"No records found for TARGET_SCENE_TYPE={TARGET_SCENE_TYPE!r}. "
        "Check scene_type metadata and FloorPlan inference."
    )

print("\nUnique sample_ids:", df_target["sample_id"].nunique())
print("Unique scenes:", df_target["scene"].nunique())
print("Unique directed uid pairs:", df_target["directed_uid_pair"].nunique())
print("Unique directed object-id pairs:", df_target["directed_object_id_pair"].nunique())
print("Unique directed type pairs:", df_target["directed_type_pair"].nunique())
print("Unique unordered type pairs:", df_target["unordered_type_pair"].nunique())
print("Unique probe labels:", df_target["probe_label"].nunique())
print("Unique evidence types:", df_target["evidence_type"].nunique())

print("\nTarget scenes:")
print(sorted(df_target["scene"].unique().tolist(), key=natural_sort_key))

print("\nTarget split counts:")
print(df_target["split"].value_counts())

display(df_target[[
    "sample_id",
    "scene",
    "split",
    "subject_type",
    "object_type",
    "directed_type_pair",
    "probe_label",
    "evidence_type",
]].head())

Target scene type: bathroom
Target subset mode: target_scene_type_and_test_split
All target-scene records: 4690
Test target-scene records: 4690
Rows used for target analysis: 4690

Unique sample_ids: 4690
Unique scenes: 30
Unique directed uid pairs: 1674
Unique directed object-id pairs: 1683
Unique directed type pairs: 510
Unique unordered type pairs: 314
Unique probe labels: 8
Unique evidence types: 1

Target scenes:
['FloorPlan401', 'FloorPlan402', 'FloorPlan403', 'FloorPlan404', 'FloorPlan405', 'FloorPlan406', 'FloorPlan407', 'FloorPlan408', 'FloorPlan409', 'FloorPlan410', 'FloorPlan411', 'FloorPlan412', 'FloorPlan413', 'FloorPlan414', 'FloorPlan415', 'FloorPlan416', 'FloorPlan417', 'FloorPlan418', 'FloorPlan419', 'FloorPlan420', 'FloorPlan421', 'FloorPlan422', 'FloorPlan423', 'FloorPlan424', 'FloorPlan425', 'FloorPlan426', 'FloorPlan427', 'FloorPlan428', 'FloorPlan429', 'FloorPlan430']

Target split counts:
split
test    4690
Name: count, dtype: int64


,sample_id,scene,split,subject_type,object_type,directed_type_pair,probe_label,evidence_type
17226,s4_1aae4c45575d78b0f1c2bf2d,FloorPlan401,test,Candle,SideTable,Candle -> SideTable,on,explicit_direct
17227,s4_bbd4d9c38b64adf793b432dc,FloorPlan401,test,DishSponge,HandTowelHolder,DishSponge -> HandTowelHolder,right_of,explicit_direct
17228,s4_a7a1d7c3e6035f6ec621f37f,FloorPlan401,test,DishSponge,SideTable,DishSponge -> SideTable,on,explicit_direct
17229,s4_c60e5639e836aae02e09ebcd,FloorPlan401,test,HandTowelHolder,PaperTowelRoll,HandTowelHolder -> PaperTowelRoll,left_of,explicit_direct
17230,s4_f0b95badbadcafa815eea80a,FloorPlan401,test,PaperTowelRoll,SideTable,PaperTowelRoll -> SideTable,on,explicit_direct


In [ ]:
# Target scene-type directed type-pair distribution

target_type_pair_distribution = (
    df_target.groupby(["subject_type", "object_type", "directed_type_pair"])
      .agg(
          num_records=("sample_id", "count"),
          num_unique_sample_ids=("sample_id", "nunique"),
          num_scenes=("scene", "nunique"),
          scenes=("scene", lambda x: ";".join(sorted(set(x), key=natural_sort_key))),
          num_directed_uid_pairs=("directed_uid_pair", "nunique"),
          num_directed_object_id_pairs=("directed_object_id_pair", "nunique"),
          num_probe_labels=("probe_label", "nunique"),
          probe_labels=("probe_label", lambda x: ";".join(sorted(set(x)))),
          num_evidence_types=("evidence_type", "nunique"),
          evidence_types=("evidence_type", lambda x: ";".join(sorted(set(x)))),
      )
      .reset_index()
      .sort_values(
          ["num_records", "num_directed_object_id_pairs", "directed_type_pair"],
          ascending=[False, False, True],
      )
)

target_total_records = len(df_target)
target_total_sample_ids = df_target["sample_id"].nunique()
target_total_type_pairs = df_target["directed_type_pair"].nunique()

target_type_pair_distribution["record_ratio_within_target"] = (
    target_type_pair_distribution["num_records"] / target_total_records
)

target_type_pair_distribution["sample_id_ratio_within_target"] = (
    target_type_pair_distribution["num_unique_sample_ids"] / target_total_sample_ids
)

target_type_pair_distribution["rank_by_records"] = (
    target_type_pair_distribution["num_records"]
    .rank(method="first", ascending=False)
    .astype(int)
)

target_type_pair_distribution["cumulative_record_ratio"] = (
    target_type_pair_distribution["record_ratio_within_target"].cumsum()
)

print("=" * 100)
print(f"{TARGET_SCENE_TYPE} directed type-pair distribution")
print("=" * 100)

display(target_type_pair_distribution.head(50))

print("\nTop concentration:")
for k in [5, 10, 20, 50]:
    top_ratio = target_type_pair_distribution.head(k)["record_ratio_within_target"].sum()
    print(f"Top {k} directed type pairs record ratio: {top_ratio:.4f}")

# Backward-compatible alias, in case later cells still use the older variable name.
bathroom_type_pair_distribution = target_type_pair_distribution

bathroom directed type-pair distribution


,subject_type,object_type,directed_type_pair,num_records,num_unique_sample_ids,num_scenes,scenes,num_directed_uid_pairs,num_directed_object_id_pairs,num_probe_labels,probe_labels,num_evidence_types,evidence_types,record_ratio_within_target,sample_id_ratio_within_target,rank_by_records,cumulative_record_ratio
144,Faucet,CounterTop,Faucet -> CounterTop,68,68,9,FloorPlan403;FloorPlan405;FloorPlan406;FloorPl...,13,12,1,on,1,explicit_direct,0.014499,0.014499,1,0.014499
360,SoapBottle,CounterTop,SoapBottle -> CounterTop,62,62,14,FloorPlan402;FloorPlan405;FloorPlan407;FloorPl...,15,14,2,on;right_of,1,explicit_direct,0.013220,0.013220,2,0.027719
181,HandTowel,HandTowelHolder,HandTowel -> HandTowelHolder,59,59,16,FloorPlan404;FloorPlan406;FloorPlan407;FloorPl...,16,16,1,on,1,explicit_direct,0.012580,0.012580,3,0.040299
102,CounterTop,SoapBottle,CounterTop -> SoapBottle,56,56,16,FloorPlan403;FloorPlan405;FloorPlan407;FloorPl...,16,16,1,supports,1,explicit_direct,0.011940,0.011940,4,0.052239
457,ToiletPaper,Toilet,ToiletPaper -> Toilet,54,54,11,FloorPlan402;FloorPlan405;FloorPlan406;FloorPl...,14,14,1,on,1,explicit_direct,0.011514,0.011514,5,0.063753
91,CounterTop,Candle,CounterTop -> Candle,52,52,13,FloorPlan405;FloorPlan412;FloorPlan414;FloorPl...,13,13,2,above;supports,1,explicit_direct,0.011087,0.011087,6,0.074840
101,CounterTop,SoapBar,CounterTop -> SoapBar,52,52,13,FloorPlan403;FloorPlan405;FloorPlan407;FloorPl...,13,13,2,above;supports,1,explicit_direct,0.011087,0.011087,7,0.085928
437,ToiletPaper,CounterTop,ToiletPaper -> CounterTop,47,47,11,FloorPlan406;FloorPlan407;FloorPlan408;FloorPl...,12,12,3,below;on;right_of,1,explicit_direct,0.010021,0.010021,8,0.095949
491,Towel,TowelHolder,Towel -> TowelHolder,44,44,12,FloorPlan402;FloorPlan403;FloorPlan406;FloorPl...,12,12,3,below;left_of;on,1,explicit_direct,0.009382,0.009382,9,0.105330
299,Sink,CounterTop,Sink -> CounterTop,42,42,8,FloorPlan403;FloorPlan406;FloorPlan408;FloorPl...,9,9,1,on,1,explicit_direct,0.008955,0.008955,10,0.114286



Top concentration:
Top 5 directed type pairs record ratio: 0.0638
Top 10 directed type pairs record ratio: 0.1143
Top 20 directed type pairs record ratio: 0.1949
Top 50 directed type pairs record ratio: 0.3674


In [ ]:
# Probe-label / relation distribution by scene type

label_distribution_by_scene_type = (
    df.groupby(["scene_type", "probe_label"])
      .agg(
          num_records=("sample_id", "count"),
          num_unique_sample_ids=("sample_id", "nunique"),
          num_scenes=("scene", "nunique"),
          num_directed_type_pairs=("directed_type_pair", "nunique"),
          num_directed_object_id_pairs=("directed_object_id_pair", "nunique"),
      )
      .reset_index()
      .sort_values(["scene_type", "num_records"], ascending=[True, False])
)

label_distribution_by_scene_type["record_ratio_within_scene_type"] = (
    label_distribution_by_scene_type["num_records"] /
    label_distribution_by_scene_type.groupby("scene_type")["num_records"].transform("sum")
)

label_distribution_by_scene_type["sample_id_ratio_within_scene_type"] = (
    label_distribution_by_scene_type["num_unique_sample_ids"] /
    label_distribution_by_scene_type.groupby("scene_type")["num_unique_sample_ids"].transform("sum")
)

print("=" * 100)
print("Probe-label distribution by scene type")
print("=" * 100)

display(label_distribution_by_scene_type)

target_label_distribution = (
    label_distribution_by_scene_type[
        label_distribution_by_scene_type["scene_type"] == TARGET_SCENE_TYPE
    ]
    .copy()
)

print(f"\n{TARGET_SCENE_TYPE} label distribution:")
display(target_label_distribution)

Probe-label distribution by scene type


,scene_type,probe_label,num_records,num_unique_sample_ids,num_scenes,num_directed_type_pairs,num_directed_object_id_pairs,record_ratio_within_scene_type,sample_id_ratio_within_scene_type
4,bathroom,left_of,1124,1124,30,274,480,0.239659,0.239659
6,bathroom,right_of,951,951,30,266,492,0.202772,0.202772
5,bathroom,on,868,868,30,51,191,0.185075,0.185075
7,bathroom,supports,649,649,30,50,179,0.138380,0.138380
0,bathroom,above,520,520,29,77,157,0.110874,0.110874
1,bathroom,below,422,422,29,64,143,0.089979,0.089979
3,bathroom,in,87,87,15,11,21,0.018550,0.018550
2,bathroom,contains,69,69,12,11,20,0.014712,0.014712
12,bedroom,left_of,885,885,28,286,526,0.195494,0.195494
14,bedroom,right_of,814,814,28,296,531,0.179810,0.179810



bathroom label distribution:


,scene_type,probe_label,num_records,num_unique_sample_ids,num_scenes,num_directed_type_pairs,num_directed_object_id_pairs,record_ratio_within_scene_type,sample_id_ratio_within_scene_type
4,bathroom,left_of,1124,1124,30,274,480,0.239659,0.239659
6,bathroom,right_of,951,951,30,266,492,0.202772,0.202772
5,bathroom,on,868,868,30,51,191,0.185075,0.185075
7,bathroom,supports,649,649,30,50,179,0.138380,0.138380
0,bathroom,above,520,520,29,77,157,0.110874,0.110874
1,bathroom,below,422,422,29,64,143,0.089979,0.089979
3,bathroom,in,87,87,15,11,21,0.018550,0.018550
2,bathroom,contains,69,69,12,11,20,0.014712,0.014712


In [ ]:
# Evidence-type distribution by scene type

evidence_distribution_by_scene_type = (
    df.groupby(["scene_type", "evidence_type"])
      .agg(
          num_records=("sample_id", "count"),
          num_unique_sample_ids=("sample_id", "nunique"),
          num_scenes=("scene", "nunique"),
          num_directed_type_pairs=("directed_type_pair", "nunique"),
          num_directed_object_id_pairs=("directed_object_id_pair", "nunique"),
      )
      .reset_index()
      .sort_values(["scene_type", "num_records"], ascending=[True, False])
)

evidence_distribution_by_scene_type["record_ratio_within_scene_type"] = (
    evidence_distribution_by_scene_type["num_records"] /
    evidence_distribution_by_scene_type.groupby("scene_type")["num_records"].transform("sum")
)

evidence_distribution_by_scene_type["sample_id_ratio_within_scene_type"] = (
    evidence_distribution_by_scene_type["num_unique_sample_ids"] /
    evidence_distribution_by_scene_type.groupby("scene_type")["num_unique_sample_ids"].transform("sum")
)

print("=" * 100)
print("Evidence-type distribution by scene type")
print("=" * 100)

display(evidence_distribution_by_scene_type)

target_evidence_distribution = (
    evidence_distribution_by_scene_type[
        evidence_distribution_by_scene_type["scene_type"] == TARGET_SCENE_TYPE
    ]
    .copy()
)

print(f"\n{TARGET_SCENE_TYPE} evidence distribution:")
display(target_evidence_distribution)

Evidence-type distribution by scene type


,scene_type,evidence_type,num_records,num_unique_sample_ids,num_scenes,num_directed_type_pairs,num_directed_object_id_pairs,record_ratio_within_scene_type,sample_id_ratio_within_scene_type
0,bathroom,explicit_direct,4690,4690,30,510,1683,1.0,1.0
1,bedroom,explicit_direct,4527,4527,28,582,1967,1.0,1.0
2,kitchen,explicit_direct,8656,8656,30,1333,4488,1.0,1.0
3,living_room,explicit_direct,4043,4043,30,473,1394,1.0,1.0



bathroom evidence distribution:


,scene_type,evidence_type,num_records,num_unique_sample_ids,num_scenes,num_directed_type_pairs,num_directed_object_id_pairs,record_ratio_within_scene_type,sample_id_ratio_within_scene_type
0,bathroom,explicit_direct,4690,4690,30,510,1683,1.0,1.0


In [ ]:
# Train vs target-scene coverage analysis:
#   How much of the target scene's probe-pair distribution
#   is covered by training scenes?


df_train = df[df["is_step6_train_scene"]].copy()
df_test = df[df["is_step6_test_scene"]].copy()

if df_train.empty:
    print("Warning: df_train is empty. Coverage analysis cannot be computed.")
else:
    train_directed_type_pairs = set(df_train["directed_type_pair"])
    train_unordered_type_pairs = set(df_train["unordered_type_pair"])
    train_subject_types = set(df_train["subject_type"])
    train_object_types = set(df_train["object_type"])
    train_probe_labels = set(df_train["probe_label"])

    target_directed_type_pairs = set(df_target["directed_type_pair"])
    target_unordered_type_pairs = set(df_target["unordered_type_pair"])
    target_subject_types = set(df_target["subject_type"])
    target_object_types = set(df_target["object_type"])
    target_probe_labels = set(df_target["probe_label"])

    coverage_summary = {
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "target_scene_type": TARGET_SCENE_TYPE,

        "num_train_records": int(len(df_train)),
        "num_target_records": int(len(df_target)),

        "num_train_scenes": int(df_train["scene"].nunique()),
        "num_target_scenes": int(df_target["scene"].nunique()),

        "num_train_directed_type_pairs": int(len(train_directed_type_pairs)),
        "num_target_directed_type_pairs": int(len(target_directed_type_pairs)),
        "num_target_directed_type_pairs_seen_in_train": int(len(target_directed_type_pairs & train_directed_type_pairs)),
        "num_target_directed_type_pairs_unseen_in_train": int(len(target_directed_type_pairs - train_directed_type_pairs)),
        "directed_type_pair_coverage_ratio": safe_ratio(
            len(target_directed_type_pairs & train_directed_type_pairs),
            len(target_directed_type_pairs),
        ),

        "num_train_unordered_type_pairs": int(len(train_unordered_type_pairs)),
        "num_target_unordered_type_pairs": int(len(target_unordered_type_pairs)),
        "num_target_unordered_type_pairs_seen_in_train": int(len(target_unordered_type_pairs & train_unordered_type_pairs)),
        "num_target_unordered_type_pairs_unseen_in_train": int(len(target_unordered_type_pairs - train_unordered_type_pairs)),
        "unordered_type_pair_coverage_ratio": safe_ratio(
            len(target_unordered_type_pairs & train_unordered_type_pairs),
            len(target_unordered_type_pairs),
        ),

        "subject_type_coverage_ratio": safe_ratio(
            len(target_subject_types & train_subject_types),
            len(target_subject_types),
        ),

        "object_type_coverage_ratio": safe_ratio(
            len(target_object_types & train_object_types),
            len(target_object_types),
        ),

        "probe_label_coverage_ratio": safe_ratio(
            len(target_probe_labels & train_probe_labels),
            len(target_probe_labels),
        ),
    }

    df_target = df_target.copy()

    df_target["directed_type_pair_seen_in_train"] = (
        df_target["directed_type_pair"].isin(train_directed_type_pairs)
    )

    df_target["unordered_type_pair_seen_in_train"] = (
        df_target["unordered_type_pair"].isin(train_unordered_type_pairs)
    )

    df_target["subject_type_seen_in_train"] = (
        df_target["subject_type"].isin(train_subject_types)
    )

    df_target["object_type_seen_in_train"] = (
        df_target["object_type"].isin(train_object_types)
    )

    df_target["probe_label_seen_in_train"] = (
        df_target["probe_label"].isin(train_probe_labels)
    )

    coverage_summary.update({
        "sample_level_directed_type_pair_seen_ratio": float(
            df_target["directed_type_pair_seen_in_train"].mean()
        ),
        "sample_level_unordered_type_pair_seen_ratio": float(
            df_target["unordered_type_pair_seen_in_train"].mean()
        ),
        "sample_level_subject_type_seen_ratio": float(
            df_target["subject_type_seen_in_train"].mean()
        ),
        "sample_level_object_type_seen_ratio": float(
            df_target["object_type_seen_in_train"].mean()
        ),
        "sample_level_probe_label_seen_ratio": float(
            df_target["probe_label_seen_in_train"].mean()
        ),
    })

    print("=" * 100)
    print(f"Train vs {TARGET_SCENE_TYPE} coverage summary")
    print("=" * 100)

    print(json.dumps(make_json_safe(coverage_summary), indent=2, ensure_ascii=False))

    unseen_target_directed_type_pairs = (
        target_type_pair_distribution[
            ~target_type_pair_distribution["directed_type_pair"].isin(train_directed_type_pairs)
        ]
        .copy()
        .sort_values("num_records", ascending=False)
    )

    seen_target_directed_type_pairs = (
        target_type_pair_distribution[
            target_type_pair_distribution["directed_type_pair"].isin(train_directed_type_pairs)
        ]
        .copy()
        .sort_values("num_records", ascending=False)
    )

    print(f"\nUnseen {TARGET_SCENE_TYPE} directed type pairs in train:")
    display(unseen_target_directed_type_pairs.head(50))

    print(f"\nSeen {TARGET_SCENE_TYPE} directed type pairs in train:")
    display(seen_target_directed_type_pairs.head(50))

    unseen_bathroom_directed_type_pairs = unseen_target_directed_type_pairs
    seen_bathroom_directed_type_pairs = seen_target_directed_type_pairs

Train vs bathroom coverage summary
{
  "model_name": "Qwen/Qwen2.5-3B",
  "model_tag": "qwen2_5_0_3b",
  "target_scene_type": "bathroom",
  "num_train_records": 17226,
  "num_target_records": 4690,
  "num_train_scenes": 88,
  "num_target_scenes": 30,
  "num_train_directed_type_pairs": 2162,
  "num_target_directed_type_pairs": 510,
  "num_target_directed_type_pairs_seen_in_train": 59,
  "num_target_directed_type_pairs_unseen_in_train": 451,
  "directed_type_pair_coverage_ratio": 0.11568627450980393,
  "num_train_unordered_type_pairs": 1346,
  "num_target_unordered_type_pairs": 314,
  "num_target_unordered_type_pairs_seen_in_train": 42,
  "num_target_unordered_type_pairs_unseen_in_train": 272,
  "unordered_type_pair_coverage_ratio": 0.1337579617834395,
  "subject_type_coverage_ratio": 0.5555555555555556,
  "object_type_coverage_ratio": 0.5555555555555556,
  "probe_label_coverage_ratio": 1.0,
  "sample_level_directed_type_pair_seen_ratio": 0.16865671641791044,
  "sample_level_unordered_ty

,subject_type,object_type,directed_type_pair,num_records,num_unique_sample_ids,num_scenes,scenes,num_directed_uid_pairs,num_directed_object_id_pairs,num_probe_labels,probe_labels,num_evidence_types,evidence_types,record_ratio_within_target,sample_id_ratio_within_target,rank_by_records,cumulative_record_ratio
181,HandTowel,HandTowelHolder,HandTowel -> HandTowelHolder,59,59,16,FloorPlan404;FloorPlan406;FloorPlan407;FloorPl...,16,16,1,on,1,explicit_direct,0.012580,0.012580,3,0.040299
457,ToiletPaper,Toilet,ToiletPaper -> Toilet,54,54,11,FloorPlan402;FloorPlan405;FloorPlan406;FloorPl...,14,14,1,on,1,explicit_direct,0.011514,0.011514,5,0.063753
101,CounterTop,SoapBar,CounterTop -> SoapBar,52,52,13,FloorPlan403;FloorPlan405;FloorPlan407;FloorPl...,13,13,2,above;supports,1,explicit_direct,0.011087,0.011087,7,0.085928
91,CounterTop,Candle,CounterTop -> Candle,52,52,13,FloorPlan405;FloorPlan412;FloorPlan414;FloorPl...,13,13,2,above;supports,1,explicit_direct,0.011087,0.011087,6,0.074840
437,ToiletPaper,CounterTop,ToiletPaper -> CounterTop,47,47,11,FloorPlan406;FloorPlan407;FloorPlan408;FloorPl...,12,12,3,below;on;right_of,1,explicit_direct,0.010021,0.010021,8,0.095949
491,Towel,TowelHolder,Towel -> TowelHolder,44,44,12,FloorPlan402;FloorPlan403;FloorPlan406;FloorPl...,12,12,3,below;left_of;on,1,explicit_direct,0.009382,0.009382,9,0.105330
331,SoapBar,CounterTop,SoapBar -> CounterTop,42,42,9,FloorPlan407;FloorPlan409;FloorPlan410;FloorPl...,9,9,1,on,1,explicit_direct,0.008955,0.008955,11,0.123241
278,ShowerCurtain,Bathtub,ShowerCurtain -> Bathtub,41,41,7,FloorPlan401;FloorPlan405;FloorPlan407;FloorPl...,7,7,1,on,1,explicit_direct,0.008742,0.008742,13,0.140725
105,CounterTop,ToiletPaper,CounterTop -> ToiletPaper,41,41,10,FloorPlan406;FloorPlan408;FloorPlan409;FloorPl...,13,12,3,above;left_of;supports,1,explicit_direct,0.008742,0.008742,12,0.131983
103,CounterTop,SprayBottle,CounterTop -> SprayBottle,39,39,10,FloorPlan407;FloorPlan408;FloorPlan409;FloorPl...,10,10,3,above;below;supports,1,explicit_direct,0.008316,0.008316,14,0.149041



Seen bathroom directed type pairs in train:


,subject_type,object_type,directed_type_pair,num_records,num_unique_sample_ids,num_scenes,scenes,num_directed_uid_pairs,num_directed_object_id_pairs,num_probe_labels,probe_labels,num_evidence_types,evidence_types,record_ratio_within_target,sample_id_ratio_within_target,rank_by_records,cumulative_record_ratio
144,Faucet,CounterTop,Faucet -> CounterTop,68,68,9,FloorPlan403;FloorPlan405;FloorPlan406;FloorPl...,13,12,1,on,1,explicit_direct,0.014499,0.014499,1,0.014499
360,SoapBottle,CounterTop,SoapBottle -> CounterTop,62,62,14,FloorPlan402;FloorPlan405;FloorPlan407;FloorPl...,15,14,2,on;right_of,1,explicit_direct,0.013220,0.013220,2,0.027719
102,CounterTop,SoapBottle,CounterTop -> SoapBottle,56,56,16,FloorPlan403;FloorPlan405;FloorPlan407;FloorPl...,16,16,1,supports,1,explicit_direct,0.011940,0.011940,4,0.052239
299,Sink,CounterTop,Sink -> CounterTop,42,42,8,FloorPlan403;FloorPlan406;FloorPlan408;FloorPl...,9,9,1,on,1,explicit_direct,0.008955,0.008955,10,0.114286
26,Cabinet,SoapBottle,Cabinet -> SoapBottle,37,37,10,FloorPlan402;FloorPlan405;FloorPlan407;FloorPl...,14,15,3,below;left_of;right_of,1,explicit_direct,0.007889,0.007889,16,0.165032
357,SoapBottle,Cabinet,SoapBottle -> Cabinet,32,32,8,FloorPlan402;FloorPlan403;FloorPlan407;FloorPl...,13,12,4,above;below;left_of;right_of,1,explicit_direct,0.006823,0.006823,22,0.208742
27,Cabinet,SprayBottle,Cabinet -> SprayBottle,32,32,9,FloorPlan402;FloorPlan407;FloorPlan408;FloorPl...,12,10,3,below;contains;right_of,1,explicit_direct,0.006823,0.006823,23,0.215565
301,Sink,Faucet,Sink -> Faucet,30,30,8,FloorPlan401;FloorPlan406;FloorPlan419;FloorPl...,8,8,1,supports,1,explicit_direct,0.006397,0.006397,27,0.241791
99,CounterTop,Sink,CounterTop -> Sink,27,27,6,FloorPlan403;FloorPlan406;FloorPlan408;FloorPl...,8,8,1,supports,1,explicit_direct,0.005757,0.005757,36,0.295522
362,SoapBottle,Drawer,SoapBottle -> Drawer,26,26,5,FloorPlan409;FloorPlan410;FloorPlan411;FloorPl...,8,8,2,above;right_of,1,explicit_direct,0.005544,0.005544,37,0.301066


In [ ]:
# Distribution shift: train vs target scene type (bathroom)

if df_train.empty:
    print("Skipping train vs target distribution shift because df_train is empty.")
else:
    train_type_pair_counts = (
        df_train.groupby("directed_type_pair")
          .agg(
              train_num_records=("sample_id", "count"),
              train_num_scenes=("scene", "nunique"),
              train_probe_labels=("probe_label", lambda x: ";".join(sorted(set(x)))),
          )
          .reset_index()
    )

    train_type_pair_counts["train_record_ratio"] = (
        train_type_pair_counts["train_num_records"] /
        train_type_pair_counts["train_num_records"].sum()
    )

    target_type_pair_counts_simple = (
        df_target.groupby("directed_type_pair")
          .agg(
              target_num_records=("sample_id", "count"),
              target_num_scenes=("scene", "nunique"),
              target_probe_labels=("probe_label", lambda x: ";".join(sorted(set(x)))),
          )
          .reset_index()
    )

    target_type_pair_counts_simple["target_record_ratio"] = (
        target_type_pair_counts_simple["target_num_records"] /
        target_type_pair_counts_simple["target_num_records"].sum()
    )

    train_vs_target_type_pair_shift = (
        target_type_pair_counts_simple
        .merge(
            train_type_pair_counts,
            on="directed_type_pair",
            how="left",
        )
    )

    for col in [
        "train_num_records",
        "train_num_scenes",
        "train_record_ratio",
    ]:
        train_vs_target_type_pair_shift[col] = (
            train_vs_target_type_pair_shift[col].fillna(0)
        )

    train_vs_target_type_pair_shift["seen_in_train"] = (
        train_vs_target_type_pair_shift["train_num_records"] > 0
    )

    train_vs_target_type_pair_shift["target_minus_train_record_ratio"] = (
        train_vs_target_type_pair_shift["target_record_ratio"]
        - train_vs_target_type_pair_shift["train_record_ratio"]
    )

    train_vs_target_type_pair_shift = (
        train_vs_target_type_pair_shift
        .sort_values(
            ["seen_in_train", "target_record_ratio"],
            ascending=[True, False],
        )
    )

    print("=" * 100)
    print(f"Train vs {TARGET_SCENE_TYPE} directed type-pair distribution shift")
    print("=" * 100)

    display(train_vs_target_type_pair_shift.head(80))

Train vs bathroom directed type-pair distribution shift


,directed_type_pair,target_num_records,target_num_scenes,target_probe_labels,target_record_ratio,train_num_records,train_num_scenes,train_probe_labels,train_record_ratio,seen_in_train,target_minus_train_record_ratio
181,HandTowel -> HandTowelHolder,59,16,on,0.012580,0.0,0.0,NaN,0.0,False,0.012580
457,ToiletPaper -> Toilet,54,11,on,0.011514,0.0,0.0,NaN,0.0,False,0.011514
91,CounterTop -> Candle,52,13,above;supports,0.011087,0.0,0.0,NaN,0.0,False,0.011087
101,CounterTop -> SoapBar,52,13,above;supports,0.011087,0.0,0.0,NaN,0.0,False,0.011087
437,ToiletPaper -> CounterTop,47,11,below;on;right_of,0.010021,0.0,0.0,NaN,0.0,False,0.010021
...,...,...,...,...,...,...,...,...,...,...,...
323,SinkBasin -> SprayBottle,16,6,above;left_of;right_of,0.003412,0.0,0.0,NaN,0.0,False,0.003412
406,TissueBox -> CounterTop,16,3,below;on,0.003412,0.0,0.0,NaN,0.0,False,0.003412
439,ToiletPaper -> Drawer,16,4,above;right_of,0.003412,0.0,0.0,NaN,0.0,False,0.003412
441,ToiletPaper -> GarbageCan,16,4,in,0.003412,0.0,0.0,NaN,0.0,False,0.003412


In [ ]:
os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

filtered_metadata_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "step6_cross_scs_filtered_probe_pair_metadata.csv",
)

df.to_csv(filtered_metadata_csv, index=False)


scene_type_summary_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "scene_type_summary.csv",
)

scene_summary_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "scene_summary.csv",
)

target_type_pair_distribution_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    f"{TARGET_SCENE_TYPE}_directed_type_pair_distribution.csv",
)

label_distribution_by_scene_type_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "label_distribution_by_scene_type.csv",
)

evidence_distribution_by_scene_type_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "evidence_distribution_by_scene_type.csv",
)

source_file_summaries_csv = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "source_file_summaries.csv",
)

scene_type_summary.to_csv(scene_type_summary_csv, index=False)
scene_summary.to_csv(scene_summary_csv, index=False)
target_type_pair_distribution.to_csv(target_type_pair_distribution_csv, index=False)
label_distribution_by_scene_type.to_csv(label_distribution_by_scene_type_csv, index=False)
evidence_distribution_by_scene_type.to_csv(evidence_distribution_by_scene_type_csv, index=False)

pd.DataFrame(source_file_summaries).to_csv(source_file_summaries_csv, index=False)


coverage_summary_json = None
unseen_target_type_pairs_csv = None
seen_target_type_pairs_csv = None
train_vs_target_type_pair_shift_csv = None

if "coverage_summary" in globals():
    coverage_summary_json = os.path.join(
        ANALYSIS_OUTPUT_DIR,
        f"train_vs_{TARGET_SCENE_TYPE}_coverage_summary.json",
    )

    with open(coverage_summary_json, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(coverage_summary), f, ensure_ascii=False, indent=2)

if "unseen_target_directed_type_pairs" in globals():
    unseen_target_type_pairs_csv = os.path.join(
        ANALYSIS_OUTPUT_DIR,
        f"{TARGET_SCENE_TYPE}_directed_type_pairs_unseen_in_train.csv",
    )

    unseen_target_directed_type_pairs.to_csv(
        unseen_target_type_pairs_csv,
        index=False,
    )

if "seen_target_directed_type_pairs" in globals():
    seen_target_type_pairs_csv = os.path.join(
        ANALYSIS_OUTPUT_DIR,
        f"{TARGET_SCENE_TYPE}_directed_type_pairs_seen_in_train.csv",
    )

    seen_target_directed_type_pairs.to_csv(
        seen_target_type_pairs_csv,
        index=False,
    )

if "train_vs_target_type_pair_shift" in globals():
    train_vs_target_type_pair_shift_csv = os.path.join(
        ANALYSIS_OUTPUT_DIR,
        f"train_vs_{TARGET_SCENE_TYPE}_directed_type_pair_distribution_shift.csv",
    )

    train_vs_target_type_pair_shift.to_csv(
        train_vs_target_type_pair_shift_csv,
        index=False,
    )

analysis_manifest = {
    "analysis_name": "ana_s6_probe_pair_dist_xscene_cross_scs",
    "aligned_step6_script": "pilot_full_step6_ex_cross_scs_q2505b",
    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "sample_family": SAMPLE_FAMILY,
    "target_scene_type": TARGET_SCENE_TYPE,

    "statistical_object": (
        "Step5 pair-level records from a single model, filtered with "
        "the same evidence, label, and split assumptions as step6_cross_scs. "
        "Hidden-state feature arrays are not analyzed."
    ),

    "step5_pt_dir": STEP5_PT_DIR,
    "pt_file_pattern": PT_FILE_PATTERN,
    "analysis_output_dir": ANALYSIS_OUTPUT_DIR,

    "label_field": LABEL_FIELD,
    "keep_evidence_types": sorted(KEEP_EVIDENCE_TYPES),
    "drop_none_label": DROP_NONE_LABEL,
    "drop_empty_label": DROP_EMPTY_LABEL,
    "none_relation_label": NONE_RELATION_LABEL,
    "allowed_labels": None if ALLOWED_LABELS is None else sorted(ALLOWED_LABELS),

    "active_train_scenes": ACTIVE_TRAIN_SCENES,
    "active_test_scenes": ACTIVE_TEST_SCENES,

    "raw_step5_records": int(raw_record_count_total),
    "kept_after_step6_cross_scs_filter": int(kept_record_count_total),
    "dropped_by_step6_cross_scs_filter": int(dropped_record_count_total),

    "num_rows_in_metadata_df": int(len(df)),
    "num_scene_types": int(df["scene_type"].nunique()),
    "num_scenes": int(df["scene"].nunique()),

    "num_target_records": int(len(df_target)),
    "num_target_scenes": int(df_target["scene"].nunique()),
    "num_target_directed_type_pairs": int(df_target["directed_type_pair"].nunique()),

    "output_files": {
        "filtered_metadata_csv": filtered_metadata_csv,
        "scene_type_summary_csv": scene_type_summary_csv,
        "scene_summary_csv": scene_summary_csv,
        "target_type_pair_distribution_csv": target_type_pair_distribution_csv,
        "label_distribution_by_scene_type_csv": label_distribution_by_scene_type_csv,
        "evidence_distribution_by_scene_type_csv": evidence_distribution_by_scene_type_csv,
        "source_file_summaries_csv": source_file_summaries_csv,
        "coverage_summary_json": coverage_summary_json,
        "unseen_target_type_pairs_csv": unseen_target_type_pairs_csv,
        "seen_target_type_pairs_csv": seen_target_type_pairs_csv,
        "train_vs_target_type_pair_shift_csv": train_vs_target_type_pair_shift_csv,
    },
}

analysis_manifest_json = os.path.join(
    ANALYSIS_OUTPUT_DIR,
    "analysis_manifest.json",
)

with open(analysis_manifest_json, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(analysis_manifest), f, ensure_ascii=False, indent=2)

print("=" * 100)
print("Saved analysis outputs")
print("=" * 100)

print("ANALYSIS_OUTPUT_DIR:", ANALYSIS_OUTPUT_DIR)

for key, path in analysis_manifest["output_files"].items():
    if path is not None:
        print(f"- {key}: {path}")

print("- analysis_manifest_json:", analysis_manifest_json)

Saved analysis outputs
ANALYSIS_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/analysis/ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs
- filtered_metadata_csv: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/analysis/ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs/step6_cross_scs_filtered_probe_pair_metadata.csv
- scene_type_summary_csv: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/analysis/ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs/scene_type_summary.csv
- scene_summary_csv: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/analysis/ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs/scene_summary.csv
- target_type_pair_distribution_csv: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/analysis/ana_s6_probe_pair_dist_xscene_qwen2_5_3b_cross_scs/bathroom_directed_type_pair_distribution.csv
- label_distribution_b

In [ ]:
compact_scene_type_table = scene_type_summary[[
    "scene_type",
    "num_step6_filtered_records",
    "record_ratio_global",
    "num_scenes",
    "num_directed_object_id_pairs",
    "num_directed_type_pairs",
    "num_subject_types",
    "num_object_types",
    "num_probe_labels",
    "num_evidence_types",
]].copy()

compact_scene_type_table = compact_scene_type_table.sort_values(
    "num_step6_filtered_records",
    ascending=False,
)

print("=" * 100)
print("Compact scene-type comparison table")
print("Filtering: explicit_direct only + non-none directional labels")
print("=" * 100)

display(compact_scene_type_table)

print("=" * 100)
print(f"Top {TARGET_SCENE_TYPE} directed type pairs")
print("=" * 100)

display(
    target_type_pair_distribution[[
        "rank_by_records",
        "directed_type_pair",
        "num_records",
        "record_ratio_within_target",
        "cumulative_record_ratio",
        "num_scenes",
        "num_directed_object_id_pairs",
        "probe_labels",
        "evidence_types",
    ]].head(20)
)

if "coverage_summary" in globals():
    print("=" * 100)
    print(f"Coverage summary: train vs {TARGET_SCENE_TYPE}")
    print("=" * 100)

    coverage_compact = pd.DataFrame([coverage_summary])
    display(coverage_compact)

Compact scene-type comparison table
Filtering: explicit_direct only + non-none directional labels


,scene_type,num_step6_filtered_records,record_ratio_global,num_scenes,num_directed_object_id_pairs,num_directed_type_pairs,num_subject_types,num_object_types,num_probe_labels,num_evidence_types
2,kitchen,8656,0.394963,30,4488,1333,60,60,8,1
0,bathroom,4690,0.213999,30,1683,510,36,36,8,1
1,bedroom,4527,0.206561,28,1967,582,48,49,8,1
3,living_room,4043,0.184477,30,1394,473,41,41,8,1


Top bathroom directed type pairs


,rank_by_records,directed_type_pair,num_records,record_ratio_within_target,cumulative_record_ratio,num_scenes,num_directed_object_id_pairs,probe_labels,evidence_types
144,1,Faucet -> CounterTop,68,0.014499,0.014499,9,12,on,explicit_direct
360,2,SoapBottle -> CounterTop,62,0.013220,0.027719,14,14,on;right_of,explicit_direct
181,3,HandTowel -> HandTowelHolder,59,0.012580,0.040299,16,16,on,explicit_direct
102,4,CounterTop -> SoapBottle,56,0.011940,0.052239,16,16,supports,explicit_direct
457,5,ToiletPaper -> Toilet,54,0.011514,0.063753,11,14,on,explicit_direct
91,6,CounterTop -> Candle,52,0.011087,0.074840,13,13,above;supports,explicit_direct
101,7,CounterTop -> SoapBar,52,0.011087,0.085928,13,13,above;supports,explicit_direct
437,8,ToiletPaper -> CounterTop,47,0.010021,0.095949,11,12,below;on;right_of,explicit_direct
491,9,Towel -> TowelHolder,44,0.009382,0.105330,12,12,below;left_of;on,explicit_direct
299,10,Sink -> CounterTop,42,0.008955,0.114286,8,9,on,explicit_direct


Coverage summary: train vs bathroom


,model_name,model_tag,target_scene_type,num_train_records,num_target_records,num_train_scenes,num_target_scenes,num_train_directed_type_pairs,num_target_directed_type_pairs,num_target_directed_type_pairs_seen_in_train,...,num_target_unordered_type_pairs_unseen_in_train,unordered_type_pair_coverage_ratio,subject_type_coverage_ratio,object_type_coverage_ratio,probe_label_coverage_ratio,sample_level_directed_type_pair_seen_ratio,sample_level_unordered_type_pair_seen_ratio,sample_level_subject_type_seen_ratio,sample_level_object_type_seen_ratio,sample_level_probe_label_seen_ratio
0,Qwen/Qwen2.5-3B,qwen2_5_0_3b,bathroom,17226,4690,88,30,2162,510,59,...,272,0.133758,0.555556,0.555556,1.0,0.168657,0.195522,0.600426,0.575267,1.0
